## **ASR Pipeline**

In [ ]:
import os

# Read the competition data from the local workspace folder.
path = os.path.abspath("individual-test-thai-call-center-asr")
audio_dir = os.path.join(path, "audio_final", "audio")
submission_path = os.path.join(path, "sample_submission.csv")

if not os.path.isdir(audio_dir):
    raise FileNotFoundError(f"Audio directory not found: {audio_dir}")
if not os.path.isfile(submission_path):
    raise FileNotFoundError(f"Submission file not found: {submission_path}")

os.makedirs("outputs", exist_ok=True)
print("Path to competition files:", path)

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json


100%|██████████| 2.30G/2.30G [01:24<00:00, 29.2MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/individual-test-thai-call-center-asr


In [ ]:
!pip install faster-whisper pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.6 MB/s eta 0:00:00


In [ ]:
# pip install faster-whisper pandas tqdm

import os
import pandas as pd
from tqdm import tqdm
from faster_whisper import WhisperModel

AUDIO_DIR = os.path.join(path, "audio_final", "audio")
SUBMISSION_PATH = os.path.join(path, "sample_submission.csv")
OUTPUT_PATH = os.path.join("outputs", "submission_all_files_asr.csv")

MODEL_NAME = "tiny"

df = pd.read_csv(SUBMISSION_PATH)
audio_paths = df["file_name"].map(lambda name: os.path.join(AUDIO_DIR, name))
missing_files = [p for p in audio_paths if not os.path.isfile(p)]
if missing_files:
    raise FileNotFoundError(f"Missing {len(missing_files)} audio files. First: {missing_files[0]}")

model = WhisperModel(
    MODEL_NAME,
    device="cpu",
    compute_type="int8",
    cpu_threads=os.cpu_count(),
    num_workers=1
)

def transcribe_audio(audio_path, vad_filter=True, beam_size=1):
    segments, _ = model.transcribe(
        audio_path,
        language="th",
        task="transcribe",
        beam_size=beam_size,
        temperature=0,
        condition_on_previous_text=False,
        vad_filter=vad_filter,
        word_timestamps=False,
        without_timestamps=True
    )
    return "".join(segment.text for segment in segments).strip()

texts = []
failed_files = []

for audio_path in tqdm(audio_paths, desc="Transcribing every audio file"):
    try:
        text = transcribe_audio(audio_path)
        if not text:
            # Retry the same file without VAD and with a wider beam.
            text = transcribe_audio(audio_path, vad_filter=False, beam_size=5)
    except Exception as exc:
        failed_files.append((audio_path, str(exc)))
        text = ""

    if not text:
        failed_files.append((audio_path, "Model returned empty text"))
    texts.append(text)

df["text"] = texts

assert len(df) == len(pd.read_csv(SUBMISSION_PATH))
blank_mask = df["text"].isna() | df["text"].str.strip().eq("")
if failed_files or blank_mask.any():
    failed_names = df.loc[blank_mask, "file_name"].tolist()
    raise RuntimeError(
        f"ASR did not produce text for {len(failed_names)} files. "
        f"First files: {failed_names[:10]}"
    )

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH)
print("Rows transcribed by model:", len(df))

Transcribing 1 representative per group:   5%|▌         | 76/1386 [02:34<44:22,  2.03s/it]


KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Validate the completed submission produced by the previous cell.
result = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
expected = pd.read_csv(SUBMISSION_PATH)

assert list(result.columns) == ["file_name", "text"]
assert result["file_name"].tolist() == expected["file_name"].tolist()
assert result["text"].str.strip().ne("").all()

print(f"Validated {len(result)} model-transcribed rows with no blank text.")

Transcribing half only: 100%|██████████| 693/693 [25:59<00:00,  2.25s/it]

Saved: /content/submission_half_asr_copy_rest.csv
Rows: 6261
Unique keys: 1386
ASR keys: 693
Copied keys: 693
